<div style="text-align:left;">
    <span style="
        display:inline-block;
        background-color:#4169E1;
        color:white;
        padding:10px 20px;
        border-radius:8px;
        font-size:45px;
        font-weight:bold;
    ">
        XGBoost Bias Analysis
    </span>
</div>

**Author:** Sarra Chouk 

**Student ID:** 60300372

**Project:** EHR Mortality Risk Prediction  

**Dataset:** MIMIC-IV

**Date:** April 4, 2026  

---

### **Objective**
To evaluate whether the trained model performs differently across demographic groups.

### **Setup and Library Imports**

In [46]:
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from plotly.subplots import make_subplots
import plotly.graph_objects as go

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

mpl.rcParams["font.family"] = ["Arial"]
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11

### **Define Paths**

In [7]:
MODEL_ARTIFACTS_DIR = Path("../artifacts/xgboost_tuned_calibrated")
BIAS_OUTPUT_DIR = MODEL_ARTIFACTS_DIR / "bias_analysis_outputs"
BIAS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model artifacts dir:", MODEL_ARTIFACTS_DIR.resolve())
print("Bias output dir:", BIAS_OUTPUT_DIR.resolve())

Model artifacts dir: /Users/sarrachouk/Desktop/ehr-mortality-prediction/artifacts/xgboost_tuned_calibrated
Bias output dir: /Users/sarrachouk/Desktop/ehr-mortality-prediction/artifacts/xgboost_tuned_calibrated/bias_analysis_outputs


### **Load Saved Model and Prediction Artifacts**

In [8]:
model = joblib.load(MODEL_ARTIFACTS_DIR / "xgb_tuned_calibrated_model.joblib")
metadata = json.load(open(MODEL_ARTIFACTS_DIR / "metadata.json", "r", encoding="utf-8"))

best_threshold = metadata.get("best_threshold", 0.5)

test_predictions = pd.read_csv(MODEL_ARTIFACTS_DIR / "test_predictions.csv")
deployment_predictions = pd.read_csv(MODEL_ARTIFACTS_DIR / "deployment_predictions.csv")

print("Best threshold from metadata:", best_threshold)
print("Test predictions shape:", test_predictions.shape)
print("Deployment predictions shape:", deployment_predictions.shape)

Best threshold from metadata: 0.07
Test predictions shape: (83356, 3)
Deployment predictions shape: (82211, 3)


### **Load Saved Feature Datasets**

In [12]:
X_test_features = pd.read_csv(MODEL_ARTIFACTS_DIR / "X_test_used.csv")
X_deployment_features = pd.read_csv(MODEL_ARTIFACTS_DIR / "X_deployment_used.csv")

print("X_test_features shape:", X_test_features.shape)
print("X_deployment_features shape:", X_deployment_features.shape)

X_test_features shape: (83356, 49)
X_deployment_features shape: (82211, 49)


### **Identify Demographic Feature Columns**

In [13]:
GENDER_COL = "gender"
AGE_COL = "age"

race_cols = [c for c in X_test_features.columns if c.lower().startswith("race_")]

print("Gender column:", GENDER_COL)
print("Age column:", AGE_COL)
print("Detected race one-hot columns:")
print(race_cols)

if GENDER_COL not in X_test_features.columns:
    raise ValueError(f"{GENDER_COL} not found in X_test_features")

if AGE_COL not in X_test_features.columns:
    raise ValueError(f"{AGE_COL} not found in X_test_features")

if len(race_cols) == 0:
    raise ValueError("No one-hot encoded race columns were detected.")

Gender column: gender
Age column: age
Detected race one-hot columns:
['race_black', 'race_asian', 'race_hispanic_latino', 'race_other']


### **Build Fairness Analysis Tables**

In [14]:
test_bias_df = X_test_features.copy().reset_index(drop=True)
deployment_bias_df = X_deployment_features.copy().reset_index(drop=True)

test_bias_df["y_true"] = test_predictions["y_true"].values
test_bias_df["y_proba"] = test_predictions["y_proba"].values
test_bias_df["y_pred"] = test_predictions["y_pred"].values

deployment_bias_df["y_true"] = deployment_predictions["y_true"].values
deployment_bias_df["y_proba"] = deployment_predictions["y_proba"].values
deployment_bias_df["y_pred"] = deployment_predictions["y_pred"].values

print("test_bias_df shape:", test_bias_df.shape)
print("deployment_bias_df shape:", deployment_bias_df.shape)

test_bias_df shape: (83356, 52)
deployment_bias_df shape: (82211, 52)


### **Build and Reconstruct Age and Race Groups**

In [18]:
def reconstruct_race_group(df, race_columns):
    df = df.copy()

    race_matrix = df[race_columns]

    max_col = race_matrix.idxmax(axis=1)

    all_zero_mask = (race_matrix.sum(axis=1) == 0)

    df["race_group"] = max_col.str.replace(r"^race_", "", regex=True)
    df.loc[all_zero_mask, "race_group"] = "Unknown"

    return df

test_bias_df = reconstruct_race_group(test_bias_df, race_cols)
deployment_bias_df = reconstruct_race_group(deployment_bias_df, race_cols)

print("Test race groups:")
print(test_bias_df["race_group"].value_counts(dropna=False))

print("\nDeployment race groups:")
print(deployment_bias_df["race_group"].value_counts(dropna=False))

def build_age_groups(df, age_col):
    df = df.copy()
    df["age_group"] = pd.cut(
        df[age_col],
        bins=[-np.inf, 39, 59, np.inf],
        labels=["<40", "40-59", "60+"]
    )
    return df

test_bias_df = build_age_groups(test_bias_df, AGE_COL)
deployment_bias_df = build_age_groups(deployment_bias_df, AGE_COL)

print("\nTest age groups:")
print(test_bias_df["age_group"].value_counts(dropna=False))

test_bias_df[GENDER_COL] = test_bias_df[GENDER_COL].astype(str).str.strip().str.title()
deployment_bias_df[GENDER_COL] = deployment_bias_df[GENDER_COL].astype(str).str.strip().str.title()

test_bias_df["race_group"] = test_bias_df["race_group"].astype(str).str.strip().str.title()
deployment_bias_df["race_group"] = deployment_bias_df["race_group"].astype(str).str.strip().str.title()

print("\nGender groups in test:")
print(test_bias_df[GENDER_COL].value_counts(dropna=False))

print("\nRace groups in test:")
print(test_bias_df["race_group"].value_counts(dropna=False))

Test race groups:
race_group
Unknown            54784
black              13781
other               6914
hispanic_latino     4891
asian               2986
Name: count, dtype: int64

Deployment race groups:
race_group
Unknown            54411
black              13203
other               6684
hispanic_latino     4851
asian               3062
Name: count, dtype: int64

Test age groups:
age_group
60+      44118
40-59    23397
<40      15841
Name: count, dtype: int64

Gender groups in test:
gender
0    43387
1    39969
Name: count, dtype: int64

Race groups in test:
race_group
Unknown            54784
Black              13781
Other               6914
Hispanic_Latino     4891
Asian               2986
Name: count, dtype: int64


### **Define Fairness Evaluation Metrics**

In [19]:
def safe_roc_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)

def safe_pr_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_score)

def compute_group_metrics(df, group_col):
    rows = []

    for group_value, group_df in df.groupby(group_col, dropna=False):
        y_true = group_df["y_true"].values
        y_pred = group_df["y_pred"].values
        y_proba = group_df["y_proba"].values

        rows.append({
            "group_column": group_col,
            "group_value": str(group_value),
            "n_samples": int(len(group_df)),
            "positive_rate": float(np.mean(y_true)),
            "predicted_positive_rate": float(np.mean(y_pred)),
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "roc_auc": float(safe_roc_auc(y_true, y_proba)) if not np.isnan(safe_roc_auc(y_true, y_proba)) else np.nan,
            "pr_auc": float(safe_pr_auc(y_true, y_proba)) if not np.isnan(safe_pr_auc(y_true, y_proba)) else np.nan,
            "tp": int(((y_true == 1) & (y_pred == 1)).sum()),
            "fp": int(((y_true == 0) & (y_pred == 1)).sum()),
            "tn": int(((y_true == 0) & (y_pred == 0)).sum()),
            "fn": int(((y_true == 1) & (y_pred == 0)).sum()),
        })

    return pd.DataFrame(rows).sort_values("n_samples", ascending=False).reset_index(drop=True)

### **Compute Fairness Metrics on the Test Set**

In [21]:
test_gender_metrics = compute_group_metrics(test_bias_df, GENDER_COL)
test_age_metrics = compute_group_metrics(test_bias_df, "age_group")
test_race_metrics = compute_group_metrics(test_bias_df, "race_group")

print("Test fairness metrics by gender")
display(test_gender_metrics)

print("Test fairness metrics by age group")
display(test_age_metrics)

print("Test fairness metrics by race")
display(test_race_metrics)

Test fairness metrics by gender


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,gender,0,43387,0.019268,0.018416,0.971120,0.239049,0.228469,0.233639,0.870863,0.167563,191,608,41943,645
1,gender,1,39969,0.023368,0.024794,0.963697,0.239152,0.253747,0.246234,0.865109,0.187215,237,754,38281,697


Test fairness metrics by age group


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,age_group,60+,44118,0.032776,0.033614,0.949748,0.240054,0.246196,0.243086,0.814359,0.188701,356,1127,41545,1090
1,age_group,40-59,23397,0.011540,0.010685,0.982904,0.240000,0.222222,0.230769,0.889107,0.140330,60,190,22937,210
2,age_group,<40,15841,0.003409,0.003598,0.994508,0.210526,0.222222,0.216216,0.958968,0.090498,12,45,15742,42


Test fairness metrics by race


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,race_group,Unknown,54784,0.020681,0.014840,0.971086,0.222632,0.159753,0.186023,0.846653,0.144172,181,632,53019,952
1,race_group,Black,13781,0.011973,0.007909,0.983601,0.220183,0.145455,0.175182,0.881673,0.142812,24,85,13531,141
2,race_group,Other,6914,0.056407,0.114550,0.890078,0.266414,0.541026,0.357022,0.874966,0.295555,211,581,5943,179
3,race_group,Hispanic_Latino,4891,0.006338,0.003680,0.992026,0.277778,0.161290,0.204082,0.880041,0.100742,5,13,4847,26
4,race_group,Asian,2986,0.017080,0.019424,0.968185,0.120690,0.137255,0.128440,0.840545,0.108786,7,51,2884,44


### **Compute Fairness Metrics on the Deployment Set**

In [22]:
deployment_gender_metrics = compute_group_metrics(deployment_bias_df, GENDER_COL)
deployment_age_metrics = compute_group_metrics(deployment_bias_df, "age_group")
deployment_race_metrics = compute_group_metrics(deployment_bias_df, "race_group")

print("Deployment fairness metrics by gender")
display(deployment_gender_metrics)

print("Deployment fairness metrics by age group")
display(deployment_age_metrics)

print("Deployment fairness metrics by race")
display(deployment_race_metrics)

Deployment fairness metrics by gender


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,gender,0,42777,0.019637,0.017089,0.971363,0.236662,0.205952,0.220242,0.876609,0.184344,173,558,41379,667
1,gender,1,39434,0.023609,0.025257,0.962342,0.221888,0.237379,0.229372,0.863911,0.180406,221,775,37728,710


Deployment fairness metrics by age group


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,age_group,60+,43514,0.032656,0.032702,0.949488,0.226985,0.227305,0.227145,0.818581,0.185303,323,1100,40993,1098
1,age_group,40-59,23209,0.012107,0.010642,0.981904,0.218623,0.192171,0.204545,0.898805,0.159340,54,193,22735,227
2,age_group,<40,15488,0.004455,0.003680,0.994060,0.298246,0.246377,0.269841,0.935362,0.194246,17,40,15379,52


Deployment fairness metrics by race


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,race_group,Unknown,54411,0.020602,0.014299,0.971017,0.206941,0.143622,0.169563,0.850405,0.136181,161,617,52673,960
1,race_group,Black,13203,0.014088,0.007650,0.981898,0.237624,0.129032,0.167247,0.881382,0.147893,24,77,12940,162
2,race_group,Other,6684,0.051765,0.116697,0.890186,0.251282,0.566474,0.348135,0.882564,0.335476,196,584,5754,150
3,race_group,Hispanic_Latino,4851,0.010719,0.004535,0.986395,0.181818,0.076923,0.108108,0.894501,0.123890,4,18,4781,48
4,race_group,Asian,3062,0.021555,0.015023,0.969301,0.195652,0.136364,0.160714,0.892503,0.177137,9,37,2959,57


### **Save Fairness Tables**

In [23]:
test_gender_metrics.to_csv(BIAS_OUTPUT_DIR / "test_gender_metrics.csv", index=False)
test_age_metrics.to_csv(BIAS_OUTPUT_DIR / "test_age_metrics.csv", index=False)
test_race_metrics.to_csv(BIAS_OUTPUT_DIR / "test_race_metrics.csv", index=False)

deployment_gender_metrics.to_csv(BIAS_OUTPUT_DIR / "deployment_gender_metrics.csv", index=False)
deployment_age_metrics.to_csv(BIAS_OUTPUT_DIR / "deployment_age_metrics.csv", index=False)
deployment_race_metrics.to_csv(BIAS_OUTPUT_DIR / "deployment_race_metrics.csv", index=False)

print("Saved fairness metric tables.")

Saved fairness metric tables.


### **Compute Disparity Summaries**

In [24]:
def compute_disparity_summary(metrics_df, metric_cols):
    summary_rows = []

    for metric in metric_cols:
        values = metrics_df[metric].dropna()
        if len(values) == 0:
            continue

        summary_rows.append({
            "group_column": metrics_df["group_column"].iloc[0],
            "metric": metric,
            "min_value": float(values.min()),
            "max_value": float(values.max()),
            "absolute_gap": float(values.max() - values.min())
        })

    return pd.DataFrame(summary_rows)

metric_cols_for_gap = [
    "positive_rate",
    "predicted_positive_rate",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc"
]

test_gender_gap = compute_disparity_summary(test_gender_metrics, metric_cols_for_gap)
test_age_gap = compute_disparity_summary(test_age_metrics, metric_cols_for_gap)
test_race_gap = compute_disparity_summary(test_race_metrics, metric_cols_for_gap)

deployment_gender_gap = compute_disparity_summary(deployment_gender_metrics, metric_cols_for_gap)
deployment_age_gap = compute_disparity_summary(deployment_age_metrics, metric_cols_for_gap)
deployment_race_gap = compute_disparity_summary(deployment_race_metrics, metric_cols_for_gap)

print("Test gender disparity summary")
display(test_gender_gap)

print("Test age disparity summary")
display(test_age_gap)

print("Test race disparity summary")
display(test_race_gap)

Test gender disparity summary


,group_column,metric,min_value,max_value,absolute_gap
0,gender,positive_rate,0.019268,0.023368,0.004100
1,gender,predicted_positive_rate,0.018416,0.024794,0.006379
2,gender,precision,0.239049,0.239152,0.000104
3,gender,recall,0.228469,0.253747,0.025278
4,gender,f1,0.233639,0.246234,0.012595
5,gender,roc_auc,0.865109,0.870863,0.005754
6,gender,pr_auc,0.167563,0.187215,0.019652


Test age disparity summary


,group_column,metric,min_value,max_value,absolute_gap
0,age_group,positive_rate,0.003409,0.032776,0.029367
1,age_group,predicted_positive_rate,0.003598,0.033614,0.030016
2,age_group,precision,0.210526,0.240054,0.029528
3,age_group,recall,0.222222,0.246196,0.023974
4,age_group,f1,0.216216,0.243086,0.026870
5,age_group,roc_auc,0.814359,0.958968,0.144609
6,age_group,pr_auc,0.090498,0.188701,0.098203


Test race disparity summary


,group_column,metric,min_value,max_value,absolute_gap
0,race_group,positive_rate,0.006338,0.056407,0.050069
1,race_group,predicted_positive_rate,0.003680,0.114550,0.110870
2,race_group,precision,0.120690,0.277778,0.157088
3,race_group,recall,0.137255,0.541026,0.403771
4,race_group,f1,0.128440,0.357022,0.228582
5,race_group,roc_auc,0.840545,0.881673,0.041128
6,race_group,pr_auc,0.100742,0.295555,0.194812


### **Fairness Analysis Plots**

In [48]:
def relabel_gender(df, gender_col):
    df = df.copy()

    gender_map = {
        0: "Female",
        1: "Male",
        "0": "Female",
        "1": "Male"
    }

    df[gender_col] = df[gender_col].map(gender_map).fillna(df[gender_col].astype(str))
    return df

test_bias_df = relabel_gender(test_bias_df, GENDER_COL)
deployment_bias_df = relabel_gender(deployment_bias_df, GENDER_COL)

print(test_bias_df[GENDER_COL].value_counts(dropna=False))
print(deployment_bias_df[GENDER_COL].value_counts(dropna=False))

test_gender_metrics = compute_group_metrics(test_bias_df, GENDER_COL)
test_age_metrics = compute_group_metrics(test_bias_df, "age_group")
test_race_metrics = compute_group_metrics(test_bias_df, "race_group")

deployment_gender_metrics = compute_group_metrics(deployment_bias_df, GENDER_COL)
deployment_age_metrics = compute_group_metrics(deployment_bias_df, "age_group")
deployment_race_metrics = compute_group_metrics(deployment_bias_df, "race_group")

gender
Female    43387
Male      39969
Name: count, dtype: int64
gender
Female    42777
Male      39434
Name: count, dtype: int64


In [ ]:
plotly_template = "plotly_white"

common_layout = dict(
    template=plotly_template,
    font=dict(family="Arial", size=13, color="#1F2937"),
    title_font=dict(size=20, color="#111827"),
    legend_title_text="Outcome",
    margin=dict(l=50, r=40, t=80, b=50),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

print("Shared chart styling configured.")


def plot_fairness_subplots_plotly(
    gender_df,
    age_df,
    race_df,
    metric="pr_auc",
    title="Fairness Comparison Across Groups",
    filename="fairness_subplots.html"
):
    gender_df = gender_df.sort_values(metric, ascending=False)
    age_df = age_df.sort_values(metric, ascending=False)
    race_df = race_df.sort_values(metric, ascending=False)

    metric_label = (
        metric.replace("_", " ").upper()
        if metric in ["pr_auc", "roc_auc"]
        else metric.replace("_", " ").title()
    )

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=("Gender", "Age Group", "Race Group"),
        horizontal_spacing=0.08
    )

    fig.add_trace(
        go.Bar(
            x=gender_df["group_value"],
            y=gender_df[metric],
            name="Gender",
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=age_df["group_value"],
            y=age_df[metric],
            name="Age Group",
        ),
        row=1, col=2
    )

    fig.add_trace(
        go.Bar(
            x=race_df["group_value"],
            y=race_df[metric],
            name="Race Group",
        ),
        row=1, col=3
    )

    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            x=0.0,
            xanchor="left"
        ),
        showlegend=False,
        height=500,
        width=1200,
        **common_layout
    )

    fig.update_yaxes(title_text=metric_label, row=1, col=1)
    fig.update_yaxes(title_text="", row=1, col=2)
    fig.update_yaxes(title_text="", row=1, col=3)

    fig.update_xaxes(tickangle=0, row=1, col=1)   
    fig.update_xaxes(tickangle=0, row=1, col=2)   
    fig.update_xaxes(tickangle=30, row=1, col=3)  

    fig.write_html(BIAS_OUTPUT_DIR / filename)
    fig.show()

gender
Female    43387
Male      39969
Name: count, dtype: int64
gender
Female    42777
Male      39434
Name: count, dtype: int64


In [50]:
plot_fairness_subplots_plotly(
    test_gender_metrics,
    test_age_metrics,
    test_race_metrics,
    metric="pr_auc",
    title="             Test Set PR-AUC Across Gender, Age, and Race Groups",
    filename="test_pr_auc_fairness_subplots.html"
)

In [51]:
plot_fairness_subplots_plotly(
    test_gender_metrics,
    test_age_metrics,
    test_race_metrics,
    metric="recall",
    title="             Test Set Recall Across Gender, Age, and Race Groups",
    filename="test_recall_fairness_subplots.html"
)

In [52]:
plot_fairness_subplots_plotly(
    deployment_gender_metrics,
    deployment_age_metrics,
    deployment_race_metrics,
    metric="pr_auc",
    title="             Deployment Set PR-AUC Across Gender, Age, and Race Groups",
    filename="deployment_pr_auc_fairness_subplots.html"
)

In [53]:
plot_fairness_subplots_plotly(
    deployment_gender_metrics,
    deployment_age_metrics,
    deployment_race_metrics,
    metric="recall",
    title="             Deployment Set Recall Across Gender, Age, and Race Groups",
    filename="deployment_recall_fairness_subplots.html"
)